In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [11]:
!pip install -q datasets transformers sentence-transformers accelerate

In [12]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline
)

from sentence_transformers import SentenceTransformer, util

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
from datasets import load_dataset

train_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

dataset = load_dataset(
    "csv",
    data_files={"train": train_path}
)["train"]

print(dataset)
print(dataset.column_names)
print(dataset[0])

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})
['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
{'id': 1, 'prompt': "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.", 'A': "Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.", 'B': 'Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.', 'C': 'Martin Heidegger does not believe in the existence of time or that it has any effect on human 

In [14]:
def add_combined_text(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

dataset_mapped = dataset.map(add_combined_text)

q1_answer = len(dataset_mapped[51]["combined_text"])

print("Q1 Answer:", q1_answer)
print("Combined Text:", repr(dataset_mapped[51]["combined_text"]))

Q1 Answer: 614
Combined Text: 'Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.'


In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Vocabulary Size:", tokenizer.vocab_size)

Vocabulary Size: 30522


In [16]:
q3_answer = tokenizer.sep_token_id

print("Q3 Answer:", q3_answer)

Q3 Answer: 102


In [17]:
all_prompts = list(dataset["prompt"])

encoded_prompts = tokenizer(
    all_prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

q4_answer = encoded_prompts["input_ids"].shape

print("Q4 Answer:", q4_answer)

Q4 Answer: torch.Size([2000, 128])


In [18]:
hidden_size = 768
num_attention_heads = 12

q5_answer = hidden_size // num_attention_heads

print("Q5 Answer:", q5_answer)

Q5 Answer: 64


In [19]:
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

prompt_0 = dataset[0]["prompt"]

inputs_0 = tokenizer(
    prompt_0,
    return_tensors="pt"
)

with torch.no_grad():
    outputs_0 = model(**inputs_0)

q6_answer = outputs_0.last_hidden_state.shape

print("Prompt 0:", prompt_0)
print("Tokenized shape:", inputs_0["input_ids"].shape)
print("Q6 Answer:", q6_answer)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prompt 0: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Tokenized shape: torch.Size([1, 31])
Q6 Answer: torch.Size([1, 31, 768])


In [20]:
# Extract [CLS] token embedding
cls_vector = outputs_0.last_hidden_state[0, 0, :]

# First 5 float values
first_5_values = cls_vector[:5]

# Sum
q7_answer = first_5_values.sum().item()

print("First 5 values:", first_5_values)
print("Raw sum:", q7_answer)
print("Q7 Answer (rounded to 4 decimals):", round(q7_answer, 4))

First 5 values: tensor([-0.4677, -0.0754, -0.2019, -0.0071, -0.4480])
Raw sum: -1.2000964879989624
Q7 Answer (rounded to 4 decimals): -1.2001


In [21]:
from transformers import AutoModel
import torch

# Load BERT with attentions enabled
attention_model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

attention_model.eval()

# Exact string given in question
text = "Light-ion fusion is a technique."

# Tokenize
attention_inputs = tokenizer(
    text,
    return_tensors="pt"
)

# Show tokens and indices
tokens = tokenizer.convert_ids_to_tokens(
    attention_inputs["input_ids"][0]
)

print("Tokens with indices:")
for i, token in enumerate(tokens):
    print(i, token)

# Forward pass
with torch.no_grad():
    attention_outputs = attention_model(**attention_inputs)

# Last layer attention
last_layer_attention = attention_outputs.attentions[-1]

# Batch 0, Head 0
first_head_attention = last_layer_attention[0, 0]

# Find exact token index of "fusion"
fusion_index = tokens.index("fusion")

# [CLS] token index 0 pays attention to fusion token
attention_weight = first_head_attention[0, fusion_index].item()

print("\nFusion token index:", fusion_index)
print("Raw attention weight:", attention_weight)
print("Q8 Answer (rounded to 4 decimals):", round(attention_weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens with indices:
0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]

Fusion token index: 4
Raw attention weight: 0.10247287899255753
Q8 Answer (rounded to 4 decimals): 0.1025


In [22]:
from sentence_transformers import SentenceTransformer, util

# Load MiniLM model
minilm_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Row index 0
prompt_0 = dataset[0]["prompt"]
option_b_0 = dataset[0]["B"]

# Generate embeddings using .encode()
embeddings = minilm_model.encode([
    prompt_0,
    option_b_0
])

# Cosine similarity using util.cos_sim()
similarity = util.cos_sim(
    embeddings[0],
    embeddings[1]
).item()

print("Prompt 0:", prompt_0)
print("Option B:", option_b_0)
print("Raw similarity:", similarity)
print("Q9 Answer (rounded to 4 decimals):", round(similarity, 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Prompt 0: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Option B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Raw similarity: 0.7658097743988037
Q9 Answer (rounded to 4 decimals): 0.7658


In [23]:
import numpy as np
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

labels = ["A", "B", "C", "D", "E"]

# -------------------------
# MAP@3 scoring function
# -------------------------
def map3_score(actual_answers, predictions):
    scores = []

    for actual, pred in zip(actual_answers, predictions):
        if actual == pred[0]:
            scores.append(1.0)
        elif actual == pred[1]:
            scores.append(0.5)
        elif actual == pred[2]:
            scores.append(1.0 / 3.0)
        else:
            scores.append(0.0)

    return np.mean(scores)


# =========================
# PIPELINE 1: TF-IDF
# =========================
tfidf_top3_all = []

for row in dataset:
    texts = [
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ]

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(texts)

    prompt_vector = tfidf_matrix[0]
    option_vectors = tfidf_matrix[1:]

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    ranked_indices = np.argsort(similarities)[::-1]

    top3 = [
        labels[i]
        for i in ranked_indices[:3]
    ]

    tfidf_top3_all.append(top3)

print("TF-IDF pipeline completed")


# =========================
# PIPELINE 2: MiniLM
# =========================
prompts = list(dataset["prompt"])

all_options = []

for row in dataset:
    for label in labels:
        all_options.append(row[label])

prompt_embeddings = minilm_model.encode(
    prompts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_tensor=True
)

option_embeddings_flat = minilm_model.encode(
    all_options,
    batch_size=64,
    show_progress_bar=True,
    convert_to_tensor=True
)

option_embeddings = option_embeddings_flat.reshape(
    len(dataset),
    5,
    -1
)

minilm_top3_all = []

for i in range(len(dataset)):
    similarities = util.cos_sim(
        prompt_embeddings[i],
        option_embeddings[i]
    )[0]

    ranked_indices = torch.argsort(
        similarities,
        descending=True
    ).tolist()

    top3 = [
        labels[j]
        for j in ranked_indices[:3]
    ]

    minilm_top3_all.append(top3)

print("MiniLM pipeline completed")


# =========================
# FINAL ANSWERS
# =========================
actual_answers = list(dataset["answer"])

minilm_map3 = map3_score(
    actual_answers,
    minilm_top3_all
)

improvement_count = 0

for actual, tfidf_pred, minilm_pred in zip(
    actual_answers,
    tfidf_top3_all,
    minilm_top3_all
):
    if (
        actual not in tfidf_pred
        and actual in minilm_pred
    ):
        improvement_count += 1

print("\nQ10 First Answer - MiniLM MAP@3:", minilm_map3)
print("Q10 Second Answer - Exact Count:", improvement_count)

TF-IDF pipeline completed


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

MiniLM pipeline completed

Q10 First Answer - MiniLM MAP@3: 0.4230833333333333
Q10 Second Answer - Exact Count: 564


In [24]:
from sentence_transformers import SentenceTransformer, util

minilm_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

prompt_0 = dataset[0]["prompt"]
option_b_0 = dataset[0]["B"]

embeddings = minilm_model.encode([
    prompt_0,
    option_b_0
])

similarity = util.cos_sim(
    embeddings[0],
    embeddings[1]
).item()

print("Raw similarity:", similarity)
print("Q9 Answer:", round(similarity, 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Raw similarity: 0.7658097743988037
Q9 Answer: 0.7658


In [25]:
from transformers import pipeline

zero_shot = pipeline(
    "zero-shot-classification"
)

prompt_1 = dataset[1]["prompt"]

candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]

result_softmax = zero_shot(
    prompt_1,
    candidate_labels=candidate_labels
)

print("Labels:", result_softmax["labels"])
print("Scores:", result_softmax["scores"])

q11_answer = result_softmax["scores"][0]

print("Q11 Answer:", round(q11_answer, 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Labels: ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to impl

In [26]:
result_multilabel = zero_shot(
    prompt_1,
    candidate_labels=candidate_labels,
    multi_label=True
)

softmax_sum = sum(result_softmax["scores"])
sigmoid_sum = sum(result_multilabel["scores"])

absolute_difference = abs(
    softmax_sum - sigmoid_sum
)

print("Softmax scores:", result_softmax["scores"])
print("Softmax sum:", softmax_sum)

print("\nMulti-label scores:", result_multilabel["scores"])
print("Multi-label sum:", sigmoid_sum)

print("\nQ12 Answer:", absolute_difference)

Softmax scores: [0.4574527442455292, 0.2750643789768219, 0.26748284697532654]
Softmax sum: 0.9999999701976776

Multi-label scores: [0.00046927088988013566, 2.0635825421777554e-05, 1.970058110600803e-05]
Multi-label sum: 0.0005096072964079212

Q12 Answer: 0.9994903629012697


In [27]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

flan_tokenizer = AutoTokenizer.from_pretrained(model_name)
flan_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

row0 = dataset[0]

exact_input = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} "
    f"or B: {row0['B']}? "
    f"Answer with just the letter A or B."
)

inputs = flan_tokenizer(
    exact_input,
    return_tensors="pt"
)

outputs = flan_model.generate(
    **inputs,
    max_new_tokens=5
)

exact_output = flan_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Exact Input:")
print(exact_input)

print("\nQ13 Exact Answer:", repr(exact_output))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Exact Input:
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.

Q13 Exact Answer: 'B'
